# Code RAG：从 AST 符号索引到依赖闭包

**面试问题：代码问答为什么不能只检索相似文本，怎样把定义、调用者和测试一起召回？**

## 回答主线

1. 代码语义围绕符号、导入、调用和继承传播，固定文本 Chunk 很容易把函数定义与调用约束拆开。
2. Code RAG 应先解析 AST 建立文件—符号—调用边，再以查询命中的符号为种子展开有限依赖闭包。
3. 检索上下文至少区分定义、直接调用者、测试和配置，不能把整个仓库无差别塞进 Prompt。
4. 每个片段要绑定 commit、文件路径、行号和内容哈希，避免跨分支混用。
5. 查询结果应打印为什么被选中以及通过哪条边到达。
6. 生产系统还需增量解析、多语言 LSP、权限和构建图。

## 真实案例

一个六文件订单服务把 `cancel_order(order_id, actor)` 改成需要 actor，但 API 调用者仍传一个参数，测试暴露 TypeError。我们用六段可执行 Python 源码构造仓库，比较词项 Top-2 与 AST 符号图展开，并故意把旧 commit 的调用者混入新定义。案例使用仓库内生成的离线脱敏小数据，输出用于学习机制，不代表生产性能。

### 输入预览：六个文件及故障问题

In [1]:
import ast  # 导入 Python AST 以解析定义、导入和调用关系。
import hashlib  # 导入哈希函数以绑定代码内容和 commit。
import re  # 导入正则表达式以做基线词项检索。

files = {  # 构造六个相互依赖的订单服务文件。
    "service.py": "def cancel_order(order_id, actor):\n    return {'order_id': order_id, 'actor': actor, 'status': 'cancelled'}\n",  # 新版核心函数要求 actor。
    "api.py": "from service import cancel_order\ndef cancel_endpoint(order_id):\n    return cancel_order(order_id)\n",  # 旧调用者漏传 actor。
    "admin.py": "from service import cancel_order\ndef admin_cancel(order_id, admin):\n    return cancel_order(order_id, admin)\n",  # 正确调用者提供两个参数。
    "test_api.py": "from api import cancel_endpoint\ndef test_cancel_endpoint():\n    assert cancel_endpoint('A7')['status'] == 'cancelled'\n",  # 测试覆盖错误 API 路径。
    "audit.py": "def record_action(actor, order_id):\n    return f'{actor}:{order_id}'\n",  # 审计辅助符号。
    "config.py": "REQUIRE_CANCEL_ACTOR = True\n",  # 配置解释签名变化原因。
}  # 完成小型代码库。
commit = "c0de43"  # 固定当前索引对应的代码版本。
question = "cancel_order 为什么在 API 测试里报缺少 actor"  # 定义真实代码排障问题。
print("问题：", question)  # 输出查询。
print("文件          行数  SHA前8  首行")  # 输出代码库表头。
for path, source in files.items():  # 逐文件展示版本信息。
    digest = hashlib.sha256(source.encode("utf-8")).hexdigest()[:8]  # 计算内容摘要。
    print(f"{path:<13} {len(source.splitlines()):>4}  {digest}  {source.splitlines()[0]}")  # 展示六个文件。

问题： cancel_order 为什么在 API 测试里报缺少 actor
文件          行数  SHA前8  首行
service.py       2  8ba0b6fe  def cancel_order(order_id, actor):
api.py           3  6e08ea8c  from service import cancel_order
admin.py         3  0ee09fbb  from service import cancel_order
test_api.py      3  b396fb2a  from api import cancel_endpoint
audit.py         2  ea5fbebf  def record_action(actor, order_id):
config.py        1  17af84bb  REQUIRE_CANCEL_ACTOR = True


## Baseline 基线：按查询词重叠取两个文件

In [2]:
def lexical_score(query, source):  # 计算查询和源码标识符词项重叠。
    query_terms = set(re.findall(r"[a-zA-Z_]+", query.lower()))  # 提取查询英文标识符。
    source_terms = set(re.findall(r"[a-zA-Z_]+", source.lower()))  # 提取源码英文标识符。
    return len(query_terms.intersection(source_terms))  # 返回简单重叠数。

baseline_ranking = sorted(files, key=lambda path: (-lexical_score(question, files[path]), path))  # 对六文件按词项稳定排序。
baseline_context = baseline_ranking[:2]  # 固定检索预算只取两个文件。
print("Baseline 排名：")  # 输出文本检索中间量。
for path in baseline_ranking:  # 逐文件展示重叠分数。
    print(f"{path:<13} score={lexical_score(question, files[path])}")  # 展示 test_api 与 service 可能挤掉 api 调用者。
print("Top-2 上下文：", baseline_context)  # 展示有限预算下的文件集合。
print("是否同时包含定义、故障调用者和测试：", {"service.py", "api.py", "test_api.py"}.issubset(set(baseline_context)))  # 直接衡量诊断闭包。

Baseline 排名：
service.py    score=2
admin.py      score=1
api.py        score=1
audit.py      score=1
test_api.py   score=1
config.py     score=0
Top-2 上下文： ['service.py', 'admin.py']
是否同时包含定义、故障调用者和测试： False


### 核心实现：AST 符号表与调用边

In [3]:
def parse_file(path, source):  # 解析单文件的定义、导入和调用。
    tree = ast.parse(source)  # 构建语法树而不执行源码。
    definitions = {}  # 保存符号到定义行号。
    imports = {}  # 保存本地名称到来源模块符号。
    calls = []  # 保存调用者到被调用名称的边。
    current_function = "<module>"  # 跟踪遍历时所在函数。

    class Visitor(ast.NodeVisitor):  # 定义带函数上下文的 AST 访问器。
        def visit_FunctionDef(self, node):  # 处理函数定义节点。
            nonlocal current_function  # 允许更新外层函数上下文。
            definitions[node.name] = node.lineno  # 记录符号和定义行。
            previous = current_function  # 保存进入前上下文。
            current_function = node.name  # 设置当前调用者符号。
            self.generic_visit(node)  # 继续访问函数体中的调用。
            current_function = previous  # 离开函数后恢复上下文。

        def visit_ImportFrom(self, node):  # 处理 from module import symbol。
            for alias in node.names:  # 遍历同一语句导入的所有名称。
                imports[alias.asname or alias.name] = {"module": node.module, "symbol": alias.name, "line": node.lineno}  # 保存来源和行号。
            self.generic_visit(node)  # 继续遍历子节点。

        def visit_Call(self, node):  # 处理函数调用节点。
            if isinstance(node.func, ast.Name):  # 只处理本例直接名称调用。
                calls.append({"caller": current_function, "callee": node.func.id, "line": node.lineno, "argc": len(node.args)})  # 保存调用参数数量。
            self.generic_visit(node)  # 继续遍历嵌套调用。

    Visitor().visit(tree)  # 执行完整 AST 遍历。
    return {"path": path, "definitions": definitions, "imports": imports, "calls": calls}  # 返回结构化文件索引。

parsed = {path: parse_file(path, source) for path, source in files.items()}  # 解析全部六个文件。
symbol_definitions = {}  # 建立全局符号定义索引。
for path, record in parsed.items():  # 遍历每个文件解析结果。
    module = path.removesuffix(".py")  # 将文件路径转换为模块名。
    for symbol, line in record["definitions"].items():  # 遍历定义符号。
        symbol_definitions[(module, symbol)] = {"path": path, "line": line}  # 保存模块限定定义位置。
print("service.py 定义：", parsed["service.py"]["definitions"])  # 展示函数签名种子。
print("api.py imports：", parsed["api.py"]["imports"])  # 展示跨文件符号来源。
print("api.py calls：", parsed["api.py"]["calls"])  # 展示故障调用者和 argc=1。

service.py 定义： {'cancel_order': 1}
api.py imports： {'cancel_order': {'module': 'service', 'symbol': 'cancel_order', 'line': 1}}
api.py calls： [{'caller': 'cancel_endpoint', 'callee': 'cancel_order', 'line': 3, 'argc': 1}]


## 结果解读：沿定义←调用者←测试展开依赖闭包

In [4]:
def dependency_context(seed_symbol):  # 从目标符号展开定义、直接调用者和调用者测试。
    selected = []  # 保存路径、原因和边。
    definition = symbol_definitions[("service", seed_symbol)]  # 定位目标符号定义。
    selected.append({"path": definition["path"], "reason": "definition", "edge": seed_symbol})  # 加入核心定义。
    caller_symbols = []  # 保存调用目标符号的本地函数。
    for path, record in parsed.items():  # 扫描所有文件的 import 与 call。
        imported_names = {name for name, info in record["imports"].items() if info["module"] == "service" and info["symbol"] == seed_symbol}  # 找到导入目标符号的本地名称。
        for call in record["calls"]:  # 遍历当前文件调用。
            if call["callee"] in imported_names:  # 当前调用解析到目标定义。
                selected.append({"path": path, "reason": "direct-caller", "edge": f"{call['caller']}->{seed_symbol}", "argc": call["argc"]})  # 加入调用者和参数数。
                caller_symbols.append((path.removesuffix(".py"), call["caller"]))  # 保存调用者供测试层展开。
    for path, record in parsed.items():  # 第二遍扫描调用者的调用者。
        for local_name, imported in record["imports"].items():  # 遍历每个导入符号。
            if (imported["module"], imported["symbol"]) in caller_symbols:  # 导入的是上一层调用函数。
                for call in record["calls"]:  # 遍历测试或上层调用。
                    if call["callee"] == local_name:  # 当前调用进入故障调用者。
                        selected.append({"path": path, "reason": "caller-of-caller", "edge": f"{call['caller']}->{imported['symbol']}"})  # 加入测试证据。
    unique = []  # 对同一路径保留首次选择原因。
    seen = set()  # 跟踪已加入文件。
    for item in selected:  # 按定义到调用者顺序去重。
        if item["path"] not in seen:  # 当前文件尚未加入。
            unique.append(item)  # 保存上下文项。
            seen.add(item["path"])  # 标记文件已见。
    return unique  # 返回有限依赖闭包。

graph_context = dependency_context("cancel_order")  # 展开目标符号依赖。
print("依赖检索轨迹：")  # 输出每个文件为何被召回。
for item in graph_context:  # 逐项展示边与参数数量。
    print(item)  # 展示 service、api/admin 和 test_api 的到达路径。
bad_call = next(call for call in parsed["api.py"]["calls"] if call["callee"] == "cancel_order")  # 定位 API 中参数不足调用。
good_call = next(call for call in parsed["admin.py"]["calls"] if call["callee"] == "cancel_order")  # 定位正确对照调用。
print(f"诊断：service 需要2参，api传{bad_call['argc']}参，admin传{good_call['argc']}参；test_api 覆盖 api 路径。")  # 输出可直接复述的根因。
closure_complete = {"service.py", "api.py", "test_api.py"}.issubset({item["path"] for item in graph_context})  # 检查定义、故障调用者和测试是否齐全。
print(f"闭包文件数={len(graph_context)}，是否包含三类关键上下文={closure_complete}")  # 报告上下文完整性。

依赖检索轨迹：
{'path': 'service.py', 'reason': 'definition', 'edge': 'cancel_order'}
{'path': 'api.py', 'reason': 'direct-caller', 'edge': 'cancel_endpoint->cancel_order', 'argc': 1}
{'path': 'admin.py', 'reason': 'direct-caller', 'edge': 'admin_cancel->cancel_order', 'argc': 2}
{'path': 'test_api.py', 'reason': 'caller-of-caller', 'edge': 'test_cancel_endpoint->cancel_endpoint'}
诊断：service 需要2参，api传1参，admin传2参；test_api 覆盖 api 路径。
闭包文件数=4，是否包含三类关键上下文=True


## 失败案例：新定义与旧分支调用者被拼进同一 Prompt

In [5]:
old_api_source = "from service import cancel_order\ndef cancel_endpoint(order_id):\n    return cancel_order(order_id, 'legacy-api')\n"  # 构造旧 commit 中已正确适配的 API 文件。
index_records = {path: {"commit": commit, "hash": hashlib.sha256(source.encode("utf-8")).hexdigest()[:8]} for path, source in files.items()}  # 为当前文件生成版本记录。
stale_api_record = {"path": "api.py", "commit": "old991", "hash": hashlib.sha256(old_api_source.encode("utf-8")).hexdigest()[:8]}  # 构造来自旧分支的候选片段。
unsafe_context = [index_records["service.py"], stale_api_record]  # 模拟只按相似度混合不同 commit。
consistent = len({record["commit"] for record in unsafe_context}) == 1  # 检查检索上下文是否来自同一快照。
safe_context = [record for record in unsafe_context if record["commit"] == commit]  # 使用请求 commit 做版本门禁。
print("混合上下文：", unsafe_context, "commit一致=", consistent)  # 展示可能产生“代码明明正确”的错误诊断。
print("版本门禁后：", safe_context)  # 展示旧 api 片段被拒绝。
print("修正策略：查询必须携带 repo+commit；符号边、源码和测试都从同一索引快照读取，缺片段时显式报告而不是跨分支补齐。")  # 总结版本隔离。

混合上下文： [{'commit': 'c0de43', 'hash': '8ba0b6fe'}, {'path': 'api.py', 'commit': 'old991', 'hash': 'd0a05b52'}] commit一致= False
版本门禁后： [{'commit': 'c0de43', 'hash': '8ba0b6fe'}]
修正策略：查询必须携带 repo+commit；符号边、源码和测试都从同一索引快照读取，缺片段时显式报告而不是跨分支补齐。


### 生产边界与上下文 Manifest

In [6]:
context_manifest = {"repo": "order-service", "commit": commit, "seed": "service.cancel_order", "files": [item["path"] for item in graph_context], "reasons": [item["reason"] for item in graph_context], "parser": "python-ast-r1"}  # 构造可回放 Code RAG 上下文清单。
print("Context Manifest：", context_manifest)  # 展示版本、种子和选择原因。
print("生产替换点：真实系统需要 Tree-sitter/LSP、多语言解析、类继承与动态调用、增量索引、构建标签、ACL 和仓库级测试执行。")  # 明确教学 AST 边界。

Context Manifest： {'repo': 'order-service', 'commit': 'c0de43', 'seed': 'service.cancel_order', 'files': ['service.py', 'api.py', 'admin.py', 'test_api.py'], 'reasons': ['definition', 'direct-caller', 'direct-caller', 'caller-of-caller'], 'parser': 'python-ast-r1'}
生产替换点：真实系统需要 Tree-sitter/LSP、多语言解析、类继承与动态调用、增量索引、构建标签、ACL 和仓库级测试执行。


## 回归测试：最后只保护依赖闭包、根因与版本隔离

In [7]:
assert not {"service.py", "api.py", "test_api.py"}.issubset(set(baseline_context))  # 验证 Top-2 文本基线无法包含完整诊断闭包。
assert {"service.py", "api.py", "test_api.py"}.issubset({item["path"] for item in graph_context})  # 验证 AST 图召回定义、故障调用者和测试。
assert bad_call["argc"] == 1 and good_call["argc"] == 2  # 验证参数数量对照直接支撑根因。
assert not consistent and safe_context == [index_records["service.py"]]  # 验证跨 commit 混用被门禁拒绝。
assert all(record["commit"] == commit for record in index_records.values())  # 验证当前索引快照版本一致。
print("回归测试通过：文本缺口、AST 闭包、参数根因、跨分支反例和 commit 隔离均成立。")  # 用少量断言总结 Code RAG 合同。

回归测试通过：文本缺口、AST 闭包、参数根因、跨分支反例和 commit 隔离均成立。
